In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../'))

In [5]:
import warnings

import json
from numpy import random
from model.optimization import create_study, ObjectiveSuggestion, SuggestionValueType, eval_model
from model.dataset import get_dataframe

from sklearn.ensemble import ExtraTreesClassifier

DEFAULT_RANDOM_SEED = 774
random.mtrand._rand.seed(DEFAULT_RANDOM_SEED)
warnings.filterwarnings("ignore")

In [8]:
def run_tests(category: str, method: str, metric: str):
  train_data = get_dataframe(category=category, dataset="train")
  pvalues = json.loads(open(f"../../preprocessed/{category}/important_genes_{method}_{metric}.json").readline())
  chosen_genes = list(set([y["gene"] for x in [sex_values[:15] for subtype_items in pvalues.values() for sex_values in subtype_items.values()] for y in x]))
  print(f"Total chosen genes: {len(chosen_genes)}")

  _, model = create_study(
    name=f"extra_trees_feature_selection_{method}_{metric}",
    model_factory=lambda **kwargs: ExtraTreesClassifier(**kwargs, class_weight="balanced", n_jobs=-1),
    custom_dataset=(train_data[chosen_genes], train_data["subtype"]),
    suggestions=[
      ObjectiveSuggestion(
        value_type=SuggestionValueType.INT,
        param="n_estimators",
        param_range=(10, 1_000)
      ),
      ObjectiveSuggestion(
        value_type=SuggestionValueType.INT,
        param="max_depth",
        param_range=(1, 100)
      ),
      ObjectiveSuggestion(
        value_type=SuggestionValueType.FLOAT,
        param="max_features",
        param_range=(1e-4, 1),
        is_log=True
      )
    ],
    report_test_results=False,
    trials=100
  )

  test_data = get_dataframe(category=category, dataset="test")
  eval_model(model, dataset=(test_data[chosen_genes], test_data["subtype"]), report=True)

In [9]:
run_tests(category="min_tpm_5", method="random_forest", metric="recall")

[I 2025-08-04 01:00:28,784] Using an existing study with name 'extra_trees_feature_selection_random_forest_recall' instead of creating a new one.


Total chosen genes: 254
   f1_weighted  f1_macro  recall_macro  balanced_accuracy  accuracy
0     0.864251  0.823348      0.816056           0.816056  0.866279


In [6]:
run_tests(category="min_tpm_5", method="random_forest", metric="f1")

[I 2025-08-03 21:37:32,748] Using an existing study with name 'extra_trees_feature_selection_random_forest_f1' instead of creating a new one.


Total chosen genes: 252
   f1_weighted  f1_macro  recall_macro  balanced_accuracy  accuracy
0     0.857736  0.816992      0.811708           0.811708  0.860465


In [7]:
run_tests(category="min_tpm_5", method="logistic", metric="recall")

[I 2025-08-03 21:37:49,706] Using an existing study with name 'extra_trees_feature_selection_logistic_recall' instead of creating a new one.


Total chosen genes: 239
   f1_weighted  f1_macro  recall_macro  balanced_accuracy  accuracy
0     0.843919  0.822437      0.815658           0.815658  0.843023


In [8]:
run_tests(category="min_tpm_5", method="logistic", metric="f1")

[I 2025-08-03 21:38:07,091] Using an existing study with name 'extra_trees_feature_selection_logistic_f1' instead of creating a new one.


Total chosen genes: 237
   f1_weighted  f1_macro  recall_macro  balanced_accuracy  accuracy
0      0.84451  0.819899      0.819003           0.819003  0.843023
